# Hito 2 - Notebook 08: Modelado Baseline
## Fase 4 de CRISP-DM (inicial) - 1.5.1 y 1.5.2

Modelos base **simples** que establecen el piso de desempeno. Los datos se leen **desde la base de datos** (integracion del notebook 07).

In [1]:
import sys
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
ROOT = Path.cwd()
while not (ROOT / 'src' / 'aldimi_common.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import aldimi_common as ac
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
%matplotlib inline
sns.set_theme(style='whitegrid'); plt.rcParams['figure.figsize'] = (9, 5)
pd.set_option('display.max_columns', None)
print('Raiz del proyecto:', ROOT)

Raiz del proyecto: D:\2026-01\MachLearning\finTF


In [2]:
sys.path.insert(0, str(ROOT))
import db_infrastructure as db
db_path = str(ac.DATA_PROCESSED / ac.DB_FILE)
salud = db.fetch_all_pacientes(db_path)
stock = db.fetch_all_inventario(db_path)
print('Salud desde BD :', salud.shape)
print('Stock desde BD :', stock.shape)

Salud desde BD : (38817, 44)
Stock desde BD : (18250, 32)


## 1.5.1 Enfoque metodologico general

- **Frente Salud:** clasificacion multiclase (Bajo/Medio/Alto). Baseline: **Arbol de decision poco profundo** y **Regresion logistica**.
- **Frente Logistica:** regresion de la **demanda (consumo) acumulada** a t+7 y t+14. Baseline: **Regresion lineal** y **media movil (naive)** (la demanda futura ~ el consumo de la ventana previa).
- Particion reproducible (`random_state=42`); en clasificacion, estratificada. En regresion se usa **split cronologico** (80/20 por fecha) para no filtrar informacion del futuro.

## 1.5.2a Baseline - Clasificacion clinica

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

y = salud['Prioridad_Atencion'].astype(str)
feats_c = [c for c in ac.health_feature_columns(salud)]
X = pd.get_dummies(salud[feats_c], drop_first=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train/Test:', Xtr.shape, Xte.shape)

Train/Test: (31053, 64) (7764, 64)


In [4]:
filas = []
modelos_clf = {
    'Arbol de decision (prof=4)': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Regresion logistica': LogisticRegression(max_iter=1000, multi_class='auto'),
}
for nombre, modelo in modelos_clf.items():
    if 'logistica' in nombre.lower():
        sc = StandardScaler(with_mean=False).fit(Xtr)
        modelo.fit(sc.transform(Xtr), ytr); pred = modelo.predict(sc.transform(Xte))
    else:
        modelo.fit(Xtr, ytr); pred = modelo.predict(Xte)
    filas.append({'Modelo': nombre,
                  'Accuracy': round(accuracy_score(yte, pred), 4),
                  'F1_macro': round(f1_score(yte, pred, average='macro'), 4),
                  'F1_Alto': round(f1_score(yte == 'Alto', pred == 'Alto'), 4)})
tabla_clf = pd.DataFrame(filas)
tabla_clf

,Modelo,Accuracy,F1_macro,F1_Alto
0,Arbol de decision (prof=4),0.9308,0.9208,0.9163
1,Regresion logistica,0.9321,0.9229,0.9212


> **Conclusion baseline clasificacion:** estos modelos simples fijan el **piso** de desempeno (Accuracy y F1 de referencia). El foco esta en el **F1 de la clase Alto**. El notebook 09 (Colab) debera superar estas metricas con RF/XGBoost.

## 1.5.2b Baseline - Regresion de la demanda de insumos

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feats_r = ac.stock_feature_columns(stock)
stock_ord = stock.sort_values('Fecha').reset_index(drop=True)
baseline_ma = {ac.DEMAND_TARGET_7: 'Consumo_Prev_7d', ac.DEMAND_TARGET_14: 'Consumo_Prev_14d'}
filas_r = []
for tgt, h in [(ac.DEMAND_TARGET_7, 't+7'), (ac.DEMAND_TARGET_14, 't+14')]:
    d = stock_ord.dropna(subset=[tgt]).reset_index(drop=True)
    corte = int(len(d) * 0.8)  # split cronologico (train pasado / test futuro)
    tr, te = d.iloc[:corte], d.iloc[corte:]
    Xtr, Xte, ytr, yte = tr[feats_r].fillna(0), te[feats_r].fillna(0), tr[tgt], te[tgt]
    # Modelo lineal
    lr = LinearRegression().fit(Xtr, ytr); pred_lr = lr.predict(Xte)
    # Baseline media movil: demanda futura ~ consumo de la ventana previa
    pred_ma = te[baseline_ma[tgt]].values
    for nombre, pred in [('Regresion lineal', pred_lr), ('Media movil (naive)', pred_ma)]:
        filas_r.append({'Horizonte': h, 'Modelo': nombre,
                        'MAE': round(mean_absolute_error(yte, pred), 3),
                        'RMSE': round(np.sqrt(mean_squared_error(yte, pred)), 3),
                        'R2': round(r2_score(yte, pred), 4)})
tabla_reg = pd.DataFrame(filas_r)
tabla_reg

,Horizonte,Modelo,MAE,RMSE,R2
0,t+7,Regresion lineal,22.988,27.386,0.9301
1,t+7,Media movil (naive),13.740,17.150,0.9726
2,t+14,Regresion lineal,23.877,28.421,0.9780
3,t+14,Media movil (naive),15.168,18.993,0.9902


> **Conclusion baseline regresion:** la media movil (naive) ya captura gran parte de la senal de demanda (el consumo previo es un fuerte predictor del futuro) y la regresion lineal la mejora. Estos fijan el piso de MAE/RMSE/R2; el notebook 10 (Colab) debera superarlo con RF/XGBoost.

## Guardado de metricas baseline (para comparar en Hito 3)

In [6]:
ac.REPORTS_DIR.mkdir(parents=True, exist_ok=True)
tabla_clf.to_csv(ac.REPORTS_DIR / 'baseline_clasificacion.csv', index=False)
tabla_reg.to_csv(ac.REPORTS_DIR / 'baseline_regresion.csv', index=False)
print('Metricas baseline guardadas en reports/.')

Metricas baseline guardadas en reports/.


## Conclusiones del Hito 2

- Datos preparados, integrados en BD y consumidos desde ella.
- Baselines funcionales que establecen el piso de desempeno de ambos frentes.
- Se confirma el desbalanceo clinico (justifica SMOTE) y el error base logistico. El **modelado avanzado** (Hito 3, Colab) buscara superar estas metricas de referencia.